In [1]:
# ============================================================
# CELL 1 — Environment Setup + Verify Prerequisites
# Notebook: 08_Personalisation.ipynb
#
# PURPOSE:
#   Demonstrate that fine-tuning the global LSTM-AE on a
#   subject's own baseline improves anomaly detection AUROC
#   compared to the global model tested on that same subject.
#
#   This is the central empirical claim of the research:
#   personalised models outperform generic ones for individual
#   physiological anomaly detection.
#
# WHAT THIS NOTEBOOK DOES:
#   For each WESAD and AffectiveROAD subject (25 total):
#   1. Load global model weights from notebook 05
#   2. Fine-tune a copy on this subject's baseline sequences
#      (low lr=1e-4, 20 epochs max, early stopping patience=5)
#   3. Score stress windows through the fine-tuned model
#   4. Compare AUROC against notebook 05 global LOSO result
#
# REQUIRES:
#   - lstm_ae_global_final.pt from notebook 05 Cell 5
#   - All processed .npz checkpoint files from notebook 05 Cell 2
#   - loso_summary_updated.json from notebook 05 Cell 4c
# ============================================================

import os
import json
import numpy as np
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

BASE         = '/content/drive/MyDrive/R26_DS_012_RESEARCH_HDS/Outputs'
CKPT_05      = os.path.join(BASE, 'LSTM_AE', 'checkpoints')
MODELS_05    = os.path.join(BASE, 'LSTM_AE', 'models')
RESULTS_05   = os.path.join(BASE, 'LSTM_AE', 'results')

PERSONAL_DIR = os.path.join(BASE, 'Personalisation')
RESULTS_08   = os.path.join(PERSONAL_DIR, 'results')
MODELS_08    = os.path.join(PERSONAL_DIR, 'models')

for d in [RESULTS_08, MODELS_08]:
    os.makedirs(d, exist_ok=True)
    print(f'  Ready: {d}')

# ── Locked constants (identical to all previous notebooks) ─
N_FEATURES = 11
T          = 5
HIDDEN     = 64
N_LAYERS   = 1

# Fine-tuning specific
FINETUNE_LR      = 1e-4   # 10x smaller than original training
FINETUNE_EPOCHS  = 20     # small — subject baseline is small
FINETUNE_PATIENCE= 5      # stop early if no improvement
FINETUNE_MIN_DELTA = 0.0001
THRESHOLD_PCT    = 95

WESAD_SUBJECTS        = ['S2','S3','S4','S5','S6','S7','S8','S9',
                         'S10','S11','S13','S14','S15','S16','S17']
WESAD_STRESS_EXCLUDED = ['S3', 'S6']
AROAD_DRIVES          = ['Drv1','Drv2','Drv3','Drv5','Drv6','Drv7',
                         'Drv8','Drv9','Drv10','Drv11','Drv12','Drv13']

WESAD_EVAL = [s for s in WESAD_SUBJECTS if s not in WESAD_STRESS_EXCLUDED]
AROAD_EVAL = AROAD_DRIVES

print(f'\nFine-tuning constants:')
print(f'  lr       = {FINETUNE_LR}  (10x lower than original)')
print(f'  epochs   = {FINETUNE_EPOCHS} max')
print(f'  patience = {FINETUNE_PATIENCE}')
print(f'  Evaluable subjects: {len(WESAD_EVAL)} WESAD + {len(AROAD_EVAL)} AROAD')
print(f'  Total runs        : {len(WESAD_EVAL) + len(AROAD_EVAL)}')

# ── Verify global model ───────────────────────────────────
import torch
import torch.nn as nn

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\nDevice: {device}')

class LSTMAutoEncoder(nn.Module):
    def __init__(self, n_features=11, hidden_size=64, n_layers=1):
        super().__init__()
        self.T = T
        self.encoder = nn.LSTM(n_features, hidden_size,
                               n_layers, batch_first=True)
        self.decoder = nn.LSTM(hidden_size, hidden_size,
                               n_layers, batch_first=True)
        self.output_layer = nn.Linear(hidden_size, n_features)

    def forward(self, x):
        _, (h_n, _) = self.encoder(x)
        bottleneck   = h_n[-1]
        dec_in       = bottleneck.unsqueeze(1).repeat(1, self.T, 1)
        dec_out, _   = self.decoder(dec_in)
        return self.output_layer(dec_out)

global_model_path  = os.path.join(MODELS_05, 'lstm_ae_global_final.pt')
global_config_path = os.path.join(MODELS_05, 'lstm_ae_global_config.json')

try:
    with open(global_config_path) as f:
        cfg = json.load(f)
    print(f'\nGlobal model config:')
    print(f'  hidden_size : {cfg["hidden_size"]}')
    print(f'  final_loss  : {cfg["final_loss"]:.6f}')
    print(f'  trained on  : {cfg["training_sequences"]} sequences')

    # Test load
    test_model = LSTMAutoEncoder(cfg['n_features'],
                                  cfg['hidden_size'], cfg['n_layers'])
    test_model.load_state_dict(
        torch.load(global_model_path, map_location=device))
    test_model = test_model.to(device)
    test_model.eval()

    dummy = torch.randn(4, T, N_FEATURES).to(device)
    assert test_model(dummy).shape == dummy.shape
    print(f'  Global model loaded and verified ✅')

except Exception as e:
    print(f'  ❌ Failed: {e}')

# ── Load notebook 05 global LOSO results for comparison ───
print(f'\n{"="*60}')
print('Notebook 05 baseline results loaded for comparison:')
print('='*60)

nb05_results = {}
for sid in WESAD_EVAL:
    rpath = os.path.join(RESULTS_05, f'WESAD_{sid}_result.json')
    with open(rpath) as f:
        r = json.load(f)
    nb05_results[f'WESAD_{sid}'] = r['metrics']['AUROC']
    auroc_str = f'{r["metrics"]["AUROC"]:.4f}' \
                if r['metrics']['AUROC'] is not None else 'N/A'
    print(f'  WESAD {sid:4s}: AUROC={auroc_str}')

for drv in AROAD_EVAL:
    rpath = os.path.join(RESULTS_05, f'AROAD_{drv}_result.json')
    with open(rpath) as f:
        r = json.load(f)
    nb05_results[f'AROAD_{drv}'] = r['metrics']['AUROC']
    auroc_str = f'{r["metrics"]["AUROC"]:.4f}' \
                if r['metrics']['AUROC'] is not None else 'N/A'
    print(f'  AROAD {drv:6s}: AUROC={auroc_str}')

# ── Verify checkpoint files ───────────────────────────────
print(f'\n{"="*60}')
print('Checkpoint file verification:')
print('='*60)

all_present = True
for sid in WESAD_EVAL:
    path = os.path.join(CKPT_05, f'WESAD_{sid}_processed.npz')
    if not os.path.exists(path):
        print(f'  ❌ MISSING: WESAD_{sid}_processed.npz')
        all_present = False

for drv in AROAD_EVAL:
    path = os.path.join(CKPT_05, f'AROAD_{drv}_processed.npz')
    if not os.path.exists(path):
        print(f'  ❌ MISSING: AROAD_{drv}_processed.npz')
        all_present = False

if all_present:
    total = len(WESAD_EVAL) + len(AROAD_EVAL)
    print(f'  ✅ All {total} checkpoint files present')

print(f'\n{"="*60}')
if all_present:
    print('✅ ALL CHECKS PASSED — ready for Cell 2 (fine-tuning loop)')
else:
    print('❌ Missing files — run notebook 05 first')
print('='*60)

Mounted at /content/drive
  Ready: /content/drive/MyDrive/R26_DS_012_RESEARCH_HDS/Outputs/Personalisation/results
  Ready: /content/drive/MyDrive/R26_DS_012_RESEARCH_HDS/Outputs/Personalisation/models

Fine-tuning constants:
  lr       = 0.0001  (10x lower than original)
  epochs   = 20 max
  patience = 5
  Evaluable subjects: 13 WESAD + 12 AROAD
  Total runs        : 25

Device: cpu

Global model config:
  hidden_size : 64
  final_loss  : 0.029977
  trained on  : 1972 sequences
  Global model loaded and verified ✅

Notebook 05 baseline results loaded for comparison:
  WESAD S2  : AUROC=1.0000
  WESAD S4  : AUROC=1.0000
  WESAD S5  : AUROC=1.0000
  WESAD S7  : AUROC=1.0000
  WESAD S8  : AUROC=1.0000
  WESAD S9  : AUROC=0.8897
  WESAD S10 : AUROC=1.0000
  WESAD S11 : AUROC=1.0000
  WESAD S13 : AUROC=1.0000
  WESAD S14 : AUROC=1.0000
  WESAD S15 : AUROC=1.0000
  WESAD S16 : AUROC=1.0000
  WESAD S17 : AUROC=1.0000
  AROAD Drv1  : AUROC=0.9463
  AROAD Drv2  : AUROC=0.9737
  AROAD Drv3  : A

In [2]:
# ============================================================
# CELL 2 — Personalisation Fine-Tuning Loop
#
# For each of the 25 evaluable subjects:
#   1. Load global model weights (fresh copy each time)
#   2. Fine-tune on THIS subject's own baseline sequences
#      using low learning rate (1e-4) and few epochs (20 max)
#   3. Score stress windows through fine-tuned model
#   4. Compare AUROC against notebook 05 global result
#
# CHECKPOINT STRATEGY:
#   Result saved after every subject.
#   Re-running skips completed subjects.
#   Colab reset → Cell 1 → Cell 2 → continues.
#
# OUTPUT FILES:
#   Personalisation/results/WESAD_SN_personal.json
#   Personalisation/results/AROAD_DrvN_personal.json
#   Personalisation/results/personalisation_summary.json
# ============================================================

import os
import json
import copy
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score
from datetime import datetime

BASE         = '/content/drive/MyDrive/R26_DS_012_RESEARCH_HDS/Outputs'
CKPT_05      = os.path.join(BASE, 'LSTM_AE', 'checkpoints')
MODELS_05    = os.path.join(BASE, 'LSTM_AE', 'models')
RESULTS_05   = os.path.join(BASE, 'LSTM_AE', 'results')
RESULTS_08   = os.path.join(BASE, 'Personalisation', 'results')

T               = 5
N_FEATURES      = 11
HIDDEN          = 64
N_LAYERS        = 1
FINETUNE_LR     = 1e-4
FINETUNE_EPOCHS = 20
FINETUNE_PAT    = 5
FINETUNE_DELTA  = 0.0001
THRESHOLD_PCT   = 95

WESAD_STRESS_EXCLUDED = ['S3', 'S6']
WESAD_SUBJECTS        = ['S2','S3','S4','S5','S6','S7','S8','S9',
                         'S10','S11','S13','S14','S15','S16','S17']
AROAD_DRIVES          = ['Drv1','Drv2','Drv3','Drv5','Drv6','Drv7',
                         'Drv8','Drv9','Drv10','Drv11','Drv12','Drv13']
WESAD_EVAL = [s for s in WESAD_SUBJECTS if s not in WESAD_STRESS_EXCLUDED]
AROAD_EVAL = AROAD_DRIVES

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# ── Model definition ──────────────────────────────────────
class LSTMAutoEncoder(nn.Module):
    def __init__(self, n_features=11, hidden_size=64, n_layers=1):
        super().__init__()
        self.T = T
        self.encoder = nn.LSTM(n_features, hidden_size,
                               n_layers, batch_first=True)
        self.decoder = nn.LSTM(hidden_size, hidden_size,
                               n_layers, batch_first=True)
        self.output_layer = nn.Linear(hidden_size, n_features)

    def forward(self, x):
        _, (h_n, _) = self.encoder(x)
        bottleneck   = h_n[-1]
        dec_in       = bottleneck.unsqueeze(1).repeat(1, self.T, 1)
        dec_out, _   = self.decoder(dec_in)
        return self.output_layer(dec_out)

# ── Load global weights once ──────────────────────────────
with open(os.path.join(MODELS_05, 'lstm_ae_global_config.json')) as f:
    cfg = json.load(f)

global_state = torch.load(
    os.path.join(MODELS_05, 'lstm_ae_global_final.pt'),
    map_location=device)
print(f'Global model weights loaded ✅\n')

# ── Load notebook 05 AUROC for comparison ─────────────────
nb05_auroc = {}
for sid in WESAD_EVAL:
    with open(os.path.join(RESULTS_05, f'WESAD_{sid}_result.json')) as f:
        r = json.load(f)
    nb05_auroc[f'WESAD_{sid}'] = r['metrics']['AUROC']
for drv in AROAD_EVAL:
    with open(os.path.join(RESULTS_05, f'AROAD_{drv}_result.json')) as f:
        r = json.load(f)
    nb05_auroc[f'AROAD_{drv}'] = r['metrics']['AUROC']

# ── Scoring functions ─────────────────────────────────────
def score_windows(model, windows):
    model.eval()
    scores = []
    with torch.no_grad():
        for w in windows:
            seq = torch.tensor(w, dtype=torch.float32)
            seq = seq.unsqueeze(0).unsqueeze(0).repeat(1, T, 1).to(device)
            mse = torch.mean((seq - model(seq)) ** 2).item()
            scores.append(mse)
    return np.array(scores)

def score_sequences(model, seqs):
    model.eval()
    scores = []
    with torch.no_grad():
        for seq in seqs:
            s   = torch.tensor(seq, dtype=torch.float32).unsqueeze(0).to(device)
            mse = torch.mean((s - model(s)) ** 2).item()
            scores.append(mse)
    return np.array(scores)

def compute_metrics(stress_scores, base_scores, threshold):
    y_true   = np.concatenate([np.ones(len(stress_scores)),
                                np.zeros(len(base_scores))])
    y_scores = np.concatenate([stress_scores, base_scores])
    y_pred   = (y_scores > threshold).astype(int)
    auroc    = float(roc_auc_score(y_true, y_scores))
    return {
        'AUROC'    : auroc,
        'F1'       : float(f1_score(y_true, y_pred, zero_division=0)),
        'precision': float(precision_score(y_true, y_pred, zero_division=0)),
        'recall'   : float(recall_score(y_true, y_pred, zero_division=0)),
        'FAR'      : float(np.mean(y_pred[y_true == 0])),
        'threshold': float(threshold),
    }

def load_ckpt(name):
    return np.load(os.path.join(CKPT_05, name), allow_pickle=True)

# ── Fine-tuning function ──────────────────────────────────
def finetune(global_state_dict, base_seqs,
             lr, epochs, patience, min_delta):
    """
    Create a fresh copy of the global model and fine-tune
    it on this subject's own baseline sequences.

    Key design decisions:
    - copy.deepcopy of state dict prevents any modification
      to the global weights across runs
    - Low lr (1e-4) preserves general knowledge while
      adapting to individual patterns
    - Few epochs (20 max) prevents overfitting on small
      subject-specific datasets
    - Early stopping with patience=5 stops immediately
      when improvement plateaus
    """
    model = LSTMAutoEncoder(N_FEATURES, HIDDEN, N_LAYERS)
    model.load_state_dict(copy.deepcopy(global_state_dict))
    model = model.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    X_t    = torch.tensor(base_seqs, dtype=torch.float32)
    loader = DataLoader(TensorDataset(X_t),
                        batch_size=min(16, len(base_seqs)),
                        shuffle=True)

    model.train()
    losses     = []
    best_loss  = float('inf')
    no_improve = 0

    for epoch in range(1, epochs + 1):
        epoch_loss = 0.0
        for (batch,) in loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            loss  = criterion(model(batch), batch)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * len(batch)
        avg = epoch_loss / len(base_seqs)
        losses.append(avg)
        if best_loss - avg > min_delta:
            best_loss  = avg
            no_improve = 0
        else:
            no_improve += 1
        if no_improve >= patience:
            break

    return model, losses

# ============================================================
# FINE-TUNING LOOP
# ============================================================
all_subjects = (
    [('WESAD', sid) for sid in WESAD_EVAL] +
    [('AROAD', drv) for drv in AROAD_EVAL]
)
total = len(all_subjects)

print(f'Total personalisation runs: {total}')
print(f'Fine-tune lr={FINETUNE_LR}, '
      f'max_epochs={FINETUNE_EPOCHS}, '
      f'patience={FINETUNE_PAT}\n')
print(f'{"Subject":<12} {"Global":>7} {"Personal":>9} '
      f'{"Δ AUROC":>9} {"Epochs":>7} {"Result"}')
print('-' * 60)

for run_idx, (dataset, subject_id) in enumerate(all_subjects, 1):

    result_path = os.path.join(RESULTS_08,
                               f'{dataset}_{subject_id}_personal.json')
    key = f'{dataset}_{subject_id}'

    if os.path.exists(result_path):
        with open(result_path) as f:
            r = json.load(f)
        g  = r['global_AUROC']
        p  = r['personal_AUROC']
        d  = r['delta_AUROC']
        ep = r['finetune_epochs']
        win = '✅ improved' if d > 0.001 else \
              ('➡ same'    if abs(d) <= 0.001 else '⚠️ degraded')
        g_str = f'{g:.4f}' if g is not None else 'N/A'
        p_str = f'{p:.4f}' if p is not None else 'N/A'
        d_str = f'{d:+.4f}' if d is not None else 'N/A'
        print(f'{dataset} {subject_id:<6} {g_str:>7} {p_str:>9} '
              f'{d_str:>9} {ep:>7}  (done) {win}')
        continue

    t_start = datetime.now()

    # Load this subject's data
    ckpt_name = (f'WESAD_{subject_id}_processed.npz'
                 if dataset == 'WESAD'
                 else f'AROAD_{subject_id}_processed.npz')
    ckpt = load_ckpt(ckpt_name)

    base_seqs   = ckpt['base_seqs']    # (n_seqs, T, 11) normalised
    stress_wins = ckpt['stress_wins']  # (n_stress, 11)  normalised
    base_wins   = base_seqs[:, -1, :]  # (n_seqs, 11) for FAR scoring

    # Fine-tune on this subject's baseline
    model_ft, ft_losses = finetune(
        global_state, base_seqs,
        FINETUNE_LR, FINETUNE_EPOCHS,
        FINETUNE_PAT, FINETUNE_DELTA
    )

    # Score with fine-tuned model
    base_seq_scores = score_sequences(model_ft, base_seqs)
    threshold       = float(np.percentile(base_seq_scores, THRESHOLD_PCT))
    base_win_scores = score_windows(model_ft, base_wins)
    stress_scores   = score_windows(model_ft, stress_wins)

    # Compute metrics
    if len(stress_wins) > 0:
        metrics = compute_metrics(stress_scores,
                                  base_win_scores, threshold)
        personal_auroc = metrics['AUROC']
    else:
        metrics        = {'AUROC': None, 'F1': None,
                          'precision': None, 'recall': None,
                          'FAR': float(np.mean(base_win_scores > threshold))}
        personal_auroc = None

    global_auroc = nb05_auroc.get(key)

    if personal_auroc is not None and global_auroc is not None:
        delta = personal_auroc - global_auroc
    else:
        delta = None

    elapsed = (datetime.now() - t_start).total_seconds()

    # Print result
    g_str = f'{global_auroc:.4f}' if global_auroc is not None else 'N/A'
    p_str = f'{personal_auroc:.4f}' if personal_auroc is not None else 'N/A'
    d_str = f'{delta:+.4f}' if delta is not None else 'N/A'
    ep    = len(ft_losses)
    win   = '✅ improved' if (delta is not None and delta > 0.001) else \
            ('➡ same'    if (delta is not None and abs(delta) <= 0.001) else
             '⚠️ degraded')

    print(f'{dataset} {subject_id:<6} {g_str:>7} {p_str:>9} '
          f'{d_str:>9} {ep:>7}  {win}  ({elapsed:.1f}s)')

    # Save result
    result = {
        'dataset'         : dataset,
        'subject'         : subject_id,
        'timestamp'       : datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'elapsed_sec'     : round(elapsed, 1),
        'n_base_seqs'     : int(len(base_seqs)),
        'n_stress_wins'   : int(len(stress_wins)),
        'finetune_epochs' : int(ep),
        'finetune_lr'     : FINETUNE_LR,
        'initial_loss'    : float(ft_losses[0]),
        'final_loss'      : float(ft_losses[-1]),
        'loss_curve'      : [round(float(l), 6) for l in ft_losses],
        'threshold'       : float(threshold),
        'base_recon_mean' : float(base_seq_scores.mean()),
        'stress_recon_mean': float(stress_scores.mean()) \
                             if len(stress_wins) > 0 else None,
        'global_AUROC'    : float(global_auroc) \
                            if global_auroc is not None else None,
        'personal_AUROC'  : float(personal_auroc) \
                            if personal_auroc is not None else None,
        'delta_AUROC'     : float(delta) if delta is not None else None,
        'metrics'         : {k: float(v) if v is not None else None
                             for k, v in metrics.items()},
    }

    with open(result_path, 'w') as f:
        json.dump(result, f, indent=2)

# ============================================================
# SUMMARY
# ============================================================
print(f'\n{"="*60}')
print('Combining results...')

all_results = []
for dataset, subject_id in all_subjects:
    rpath = os.path.join(RESULTS_08,
                         f'{dataset}_{subject_id}_personal.json')
    if os.path.exists(rpath):
        with open(rpath) as f:
            all_results.append(json.load(f))

# Subjects with valid delta
deltas_wesad = [r['delta_AUROC'] for r in all_results
                if r['dataset'] == 'WESAD'
                and r['delta_AUROC'] is not None]
deltas_aroad = [r['delta_AUROC'] for r in all_results
                if r['dataset'] == 'AROAD'
                and r['delta_AUROC'] is not None]
all_deltas   = deltas_wesad + deltas_aroad

global_aurocs_all  = [r['global_AUROC']   for r in all_results
                      if r['global_AUROC'] is not None]
personal_aurocs_all= [r['personal_AUROC'] for r in all_results
                      if r['personal_AUROC'] is not None]

improved  = sum(1 for d in all_deltas if d > 0.001)
same      = sum(1 for d in all_deltas if abs(d) <= 0.001)
degraded  = sum(1 for d in all_deltas if d < -0.001)

summary = {
    'generated'             : datetime.now().strftime('%Y-%m-%d %H:%M'),
    'finetune_lr'           : FINETUNE_LR,
    'finetune_epochs_max'   : FINETUNE_EPOCHS,
    'total_subjects'        : len(all_results),
    'global_AUROC_mean'     : round(float(np.mean(global_aurocs_all)), 4),
    'personal_AUROC_mean'   : round(float(np.mean(personal_aurocs_all)), 4),
    'delta_AUROC_mean'      : round(float(np.mean(all_deltas)), 4),
    'delta_AUROC_std'       : round(float(np.std(all_deltas)), 4),
    'WESAD_delta_mean'      : round(float(np.mean(deltas_wesad)), 4),
    'AROAD_delta_mean'      : round(float(np.mean(deltas_aroad)), 4),
    'n_improved'            : improved,
    'n_same'                : same,
    'n_degraded'            : degraded,
    'individual_results'    : all_results,
}

summary_path = os.path.join(RESULTS_08, 'personalisation_summary.json')
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)

print(f'\n{"="*60}')
print('PERSONALISATION COMPLETE — Summary')
print('='*60)
print(f'\n  Global model mean AUROC   : '
      f'{summary["global_AUROC_mean"]:.4f}')
print(f'  Personal model mean AUROC : '
      f'{summary["personal_AUROC_mean"]:.4f}')
print(f'  Mean AUROC improvement    : '
      f'{summary["delta_AUROC_mean"]:+.4f} '
      f'± {summary["delta_AUROC_std"]:.4f}')
print(f'\n  WESAD  Δ AUROC mean       : '
      f'{summary["WESAD_delta_mean"]:+.4f}')
print(f'  AROAD  Δ AUROC mean       : '
      f'{summary["AROAD_delta_mean"]:+.4f}')
print(f'\n  Improved  : {improved}/{len(all_deltas)} subjects')
print(f'  Same      : {same}/{len(all_deltas)} subjects')
print(f'  Degraded  : {degraded}/{len(all_deltas)} subjects')
print(f'\n  Summary saved: personalisation_summary.json')
print(f'\n{"="*60}')
if improved >= len(all_deltas) * 0.5:
    print('✅ Personalisation improves majority of subjects.')
    print('   Phase 1 complete. Ready for dissertation write-up.')
else:
    print('⚠️  Personalisation did not improve majority of subjects.')
    print('   Investigate degraded subjects before write-up.')
print('='*60)

Device: cpu
Global model weights loaded ✅

Total personalisation runs: 25
Fine-tune lr=0.0001, max_epochs=20, patience=5

Subject       Global  Personal   Δ AUROC  Epochs Result
------------------------------------------------------------
WESAD S2      1.0000    1.0000   +0.0000      20  (done) ➡ same
WESAD S4      1.0000    1.0000   +0.0000      20  (done) ➡ same
WESAD S5      1.0000    1.0000   +0.0000      20  (done) ➡ same
WESAD S7      1.0000    1.0000   +0.0000      20  (done) ➡ same
WESAD S8      1.0000    1.0000   +0.0000      20  (done) ➡ same
WESAD S9      0.8897    0.9294   +0.0397      20  (done) ✅ improved
WESAD S10     1.0000    1.0000   +0.0000      20  (done) ➡ same
WESAD S11     1.0000    1.0000   +0.0000      20  (done) ➡ same
WESAD S13     1.0000    1.0000   +0.0000      20  (done) ➡ same
WESAD S14     1.0000    1.0000   +0.0000      20  (done) ➡ same
WESAD S15     1.0000    1.0000   +0.0000      20  (done) ➡ same
WESAD S16     1.0000    1.0000   +0.0000      20  (do